<a href="https://colab.research.google.com/github/wykyty/learning-journey/blob/main/01-sparse-autoencoders/03_training_batchtopk_sae_t5_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a BatchTopK SAE on T5-large Decoder with SAELens

This notebook trains a **BatchTopK** Sparse Autoencoder on the **decoder** side of T5-large using SAELens's `BatchTopKTrainingSAE`.

BatchTopK differs from standard (ReLU + L1) SAEs:
- Uses **global top-k selection** across the entire batch (not per-sample)
- Sparsity is controlled by `k` (average active features) instead of L1 coefficient
- Uses **auxiliary loss** to recover dead neurons instead of L1 warm-up
- At inference time, converts to JumpReLU via a learned threshold (EMA)

Since SAELens's `SAETrainingRunner` does not support `HookedEncoderDecoder`, we:
1. Load T5-large via TransformerLens's `HookedEncoderDecoder`
2. Manually collect activations from a decoder hook point
3. Train `BatchTopKTrainingSAE` with a manual training loop

## Setup

In [ ]:
try:
    import google.colab  # noqa: F401
    %pip install sae-lens transformer-lens datasets
except ImportError:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")

In [ ]:
import torch
import os
from transformer_lens import HookedEncoderDecoder
from sae_lens.saes.batchtopk_sae import BatchTopKTrainingSAE, BatchTopKTrainingSAEConfig
from sae_lens.saes.sae import TrainStepInput

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Load T5-large via TransformerLens

In [ ]:
model = HookedEncoderDecoder.from_pretrained("google-t5/t5-large")
model.eval()

print(f"Model: google-t5/t5-large")
print(f"  d_model = {model.cfg.d_model}")
print(f"  d_mlp   = {model.cfg.d_mlp}")
print(f"  n_heads = {model.cfg.n_heads}")
print(f"  n_layers = {model.cfg.n_layers}")

输出如下：
```
WARNING:root:Support for T5 in TransformerLens is currently experimental, until such a time when it has feature parity with HookedTransformer and has been tested on real research tasks. Until then, backward compatibility is not guaranteed. Please see the docs for information on the limitations of the current implementation.
If using T5 for interpretability research, keep in mind that T5 has some significant architectural differences to GPT. The major one is that T5 is an Encoder-Decoder modelAlso, it uses relative positional embeddings, different types of Attention (without bias) and LayerNorm

Loaded pretrained model google-t5/t5-large into HookedTransformer
Model: google-t5/t5-large
  d_model = 1024
  d_mlp   = 4096
  n_heads = 16
  n_layers = 24

```

## Test: Verify decoder hook activation collection

In [ ]:
tokenizer = model.tokenizer

source = "The quick brown fox jumps over the lazy dog."
target = "A fox jumps over a dog."

enc_tokens = tokenizer(source, return_tensors="pt")
dec_tokens = tokenizer(target, return_tensors="pt")

with torch.no_grad():
    logits, cache = model.run_with_cache(
        enc_tokens.input_ids,
        decoder_input=dec_tokens.input_ids,
        names_filter=lambda name: name == "decoder.12.hook_mlp_out",
    )

acts = cache["decoder.12.hook_mlp_out"]
print(f"Decoder.12 hook_mlp_out shape: {acts.shape}")
print(f"  -> (batch={acts.shape[0]}, seq_len={acts.shape[1]}, d_model={acts.shape[2]})")

输出如下：
```

WARNING:root:No attention mask provided. Assuming all tokens should be attended to.
Decoder.12 hook_mlp_out shape: torch.Size([1, 11, 1024])
  -> (batch=1, seq_len=11, d_model=1024)
```

## Load Dataset (XSum for summarization)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("EdinburghNLP/xsum", split="train", streaming=True)
dataset_iter = iter(dataset)

sample = next(dataset_iter)
print(f"Document: {sample['document'][:200]}...")
print(f"Summary:  {sample['summary']}")

输出：
```
Document: The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing w...
Summary:  Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.
```

## Config & Activation Collection

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # SAE architecture
    d_in: int = 1024          # T5-large d_model
    d_sae: int = 16384        # SAE feature count (16x expansion)
    # Hook point
    hook_name: str = "decoder.12.hook_mlp_out"
    # Data
    context_size: int = 128   # max encoder token length
    target_size: int = 64     # max decoder token length
    batch_size: int = 4096    # number of activation tokens per training step
    # BatchTopK specific
    k: float = 100.0          # avg active features per sample (float!)
    aux_loss_coefficient: float = 1.0
    rescale_acts_by_decoder_norm: bool = True
    # Training
    total_steps: int = 10_000
    lr: float = 5e-5
    log_every: int = 100
    n_batches_for_norm_estimate: int = 50

cfg = Config()
print(f"BatchTopK SAE Config: d_in={cfg.d_in}, d_sae={cfg.d_sae}, k={cfg.k}")
print(f"Hook: {cfg.hook_name}")
print(f"Expansion ratio: {cfg.d_sae // cfg.d_in}x")
print(f"Aux loss coefficient: {cfg.aux_loss_coefficient}")
print(f"Rescale by decoder norm: {cfg.rescale_acts_by_decoder_norm}")

输出：
```
BatchTopK SAE Config: d_in=1024, d_sae=16384, k=100.0
Hook: decoder.12.hook_mlp_out
Expansion ratio: 16x
Aux loss coefficient: 1.0
Rescale by decoder norm: True
```

In [ ]:
@torch.no_grad()
def collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=4096):
    """Collect decoder activations from XSum document-summary pairs."""
    collected = []
    total_tokens = 0

    while total_tokens < target_tokens:
        try:
            sample = next(dataset_iter)
        except StopIteration:
            break

        source = sample["document"]
        target = sample["summary"]

        if not source.strip() or not target.strip():
            continue

        enc_tokens = tokenizer(
            source, return_tensors="pt", truncation=True,
            max_length=cfg.context_size, padding=False,
        )
        dec_tokens = tokenizer(
            target, return_tensors="pt", truncation=True,
            max_length=cfg.target_size, padding=False,
        )
        n_dec = dec_tokens.input_ids.shape[1]

        _, cache = model.run_with_cache(
            enc_tokens.input_ids,
            decoder_input=dec_tokens.input_ids,
            names_filter=lambda name: name == cfg.hook_name,
        )

        acts = cache[cfg.hook_name].squeeze(0).float()
        collected.append(acts)
        total_tokens += acts.shape[0]

    if not collected:
        return torch.empty(0, cfg.d_in, device=device)

    all_acts = torch.cat(collected, dim=0)
    if all_acts.shape[0] > target_tokens:
        indices = torch.randperm(all_acts.shape[0])[:target_tokens]
        all_acts = all_acts[indices]
    return all_acts

# Test
test_acts = collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=512)
print(f"Collected activations shape: {test_acts.shape}")

## Create BatchTopKTrainingSAE

Key config fields:
- `k: float` — average number of active features per sample (global top-k across batch)
- `aux_loss_coefficient` — weight on dead-neuron auxiliary loss
- `rescale_acts_by_decoder_norm` — scale pre-activations by decoder row norms before top-k
- `topk_threshold_lr` — EMA rate for the JumpReLU threshold

In [ ]:
sae_cfg = BatchTopKTrainingSAEConfig(
    d_in=cfg.d_in,
    d_sae=cfg.d_sae,
    k=cfg.k,
    aux_loss_coefficient=cfg.aux_loss_coefficient,
    rescale_acts_by_decoder_norm=cfg.rescale_acts_by_decoder_norm,
    topk_threshold_lr=0.01,
    apply_b_dec_to_input=False,
    normalize_activations="expected_average_only_in",
    decoder_init_norm=0.1,
    device=device,
    dtype="float32",
)

sae = BatchTopKTrainingSAE(sae_cfg).to(device)

n_params = sum(p.numel() for p in sae.parameters())
print(f"BatchTopK SAE initialized: {n_params:,} parameters")
print(f"  W_enc: {sae.W_enc.shape}")
print(f"  W_dec: {sae.W_dec.shape}")
print(f"  b_enc: {sae.b_enc.shape}")
print(f"  b_dec: {sae.b_dec.shape}")
print(f"  topk_threshold: {sae.topk_threshold}")

## Estimate Activation Scaling Factor

In [ ]:
import numpy as np

print("Estimating activation scaling factor...")
norms = []
for i in range(cfg.n_batches_for_norm_estimate):
    batch = collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=4096)
    if batch.shape[0] > 0:
        norms.append(batch.norm(dim=-1).mean().item())
    if (i + 1) % 10 == 0:
        print(f"  Batch {i + 1}/{cfg.n_batches_for_norm_estimate}")

mean_norm = np.mean(norms)
scaling_factor = (cfg.d_in ** 0.5) / mean_norm

print(f"\nMean activation L2 norm: {mean_norm:.4f}")
print(f"Target norm (sqrt(d_in)): {cfg.d_in ** 0.5:.4f}")
print(f"Scaling factor: {scaling_factor:.4f}")

sae.scaling_factor = torch.tensor(scaling_factor, device=device)

输出：
```
Mean activation L2 norm: 1179.0550
Target norm (sqrt(d_in)): 32.0000
Scaling factor: 0.0271
```

## Training Loop

BatchTopK training differences from Standard SAE:
- **No L1 loss** — sparsity comes from top-k selection
- **MSE + aux loss** — aux loss recovers dead neurons by having them reconstruct the residual
- **`coefficients` is empty** — `BatchTopKTrainingSAE.get_coefficients()` returns `{}`
- **`topk_threshold`** is auto-updated inside `training_forward_pass`

In [ ]:
from tqdm.auto import tqdm

optimizer = torch.optim.Adam(sae.parameters(), lr=cfg.lr, betas=(0.9, 0.999))

# Metrics tracking
metrics = {
    "step": [], "total_loss": [], "mse_loss": [], "aux_loss": [],
    "explained_variance": [], "l0": [], "topk_threshold": [],
}

print(f"\nStarting BatchTopK training for {cfg.total_steps} steps...")
print(f"  Hook: {cfg.hook_name}")
print(f"  d_in={cfg.d_in}, d_sae={cfg.d_sae}, k={cfg.k}")
print(f"  Aux loss coefficient: {cfg.aux_loss_coefficient}")
print(f"  Rescale by decoder norm: {cfg.rescale_acts_by_decoder_norm}")
print(f"  Dataset: XSum (streaming)\n")

sae.train()
pbar = tqdm(range(cfg.total_steps), desc="Training BatchTopK SAE")

for step in pbar:
    # Collect activations
    sae_in = collect_activations_batch(
        dataset_iter, model, tokenizer, cfg, target_tokens=cfg.batch_size
    )
    if sae_in.shape[0] == 0:
        continue

    # Apply scaling factor
    sae_in = sae_in * sae.scaling_factor

    # Forward pass — coefficients is empty for BatchTopK (no L1)
    step_output = sae.training_forward_pass(
        step_input=TrainStepInput(
            sae_in=sae_in,
            coefficients={},  # BatchTopK uses aux_loss from config, not scheduled coefficients
            dead_neuron_mask=None,
            n_training_steps=step,
            is_logging_step=(step % cfg.log_every == 0),
        )
    )

    # Backward + optimize
    optimizer.zero_grad()
    step_output.loss.backward()
    torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
    optimizer.step()

    # Compute metrics
    with torch.no_grad():
        l0 = step_output.feature_acts.bool().float().sum(-1).mean().item()
        per_token_l2 = (step_output.sae_out - sae_in).pow(2).sum(dim=-1)
        total_var = (sae_in - sae_in.mean(0)).pow(2).sum(-1)
        ev = 1 - per_token_l2.mean() / (total_var.mean() + 1e-8)

    # Track metrics
    metrics["step"].append(step)
    metrics["total_loss"].append(step_output.loss.item())
    metrics["mse_loss"].append(step_output.losses["mse_loss"].item())
    metrics["aux_loss"].append(step_output.losses["auxiliary_reconstruction_loss"].item())
    metrics["explained_variance"].append(ev.item())
    metrics["l0"].append(l0)
    metrics["topk_threshold"].append(sae.topk_threshold.item())

    if step % cfg.log_every == 0:
        pbar.set_postfix({
            "loss": f"{step_output.loss.item():.4f}",
            "mse": f"{step_output.losses['mse_loss'].item():.4f}",
            "aux": f"{step_output.losses['auxiliary_reconstruction_loss'].item():.4f}",
            "ev": f"{ev.item():.4f}",
            "l0": f"{l0:.1f}",
        })

pbar.close()
print("\nTraining complete!")

## Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("T5-large Decoder BatchTopK SAE Training Metrics", fontsize=14)

axes[0, 0].plot(metrics["step"], metrics["total_loss"])
axes[0, 0].set_title("Total Loss (MSE + Aux)")
axes[0, 0].set_xlabel("Step")

axes[0, 1].plot(metrics["step"], metrics["mse_loss"])
axes[0, 1].set_title("MSE Loss")
axes[0, 1].set_xlabel("Step")

axes[0, 2].plot(metrics["step"], metrics["aux_loss"])
axes[0, 2].set_title("Auxiliary Loss (Dead Neuron Recovery)")
axes[0, 2].set_xlabel("Step")

axes[1, 0].plot(metrics["step"], metrics["explained_variance"])
axes[1, 0].set_title("Explained Variance")
axes[1, 0].set_xlabel("Step")
axes[1, 0].axhline(y=0.8, color="r", linestyle="--", alpha=0.5, label="Target (0.8)")
axes[1, 0].legend()

axes[1, 1].plot(metrics["step"], metrics["l0"])
axes[1, 1].set_title("L0 (Active Features per Token)")
axes[1, 1].set_xlabel("Step")
axes[1, 1].axhline(y=cfg.k, color="r", linestyle="--", alpha=0.5, label=f"Target k={cfg.k}")
axes[1, 1].legend()

axes[1, 2].plot(metrics["step"], metrics["topk_threshold"])
axes[1, 2].set_title("TopK Threshold (EMA)")
axes[1, 2].set_xlabel("Step")

plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
sae.eval()

all_feature_acts = []
all_reconstructions = []
all_inputs = []

with torch.no_grad():
    for _ in range(10):
        batch = collect_activations_batch(
            dataset_iter, model, tokenizer, cfg, target_tokens=4096
        )
        if batch.shape[0] == 0:
            continue
        scaled = batch * sae.scaling_factor
        feature_acts, hidden_pre = sae.encode_with_hidden_pre(scaled)
        reconstruction = sae.decode(feature_acts)

        all_feature_acts.append(feature_acts)
        all_reconstructions.append(reconstruction)
        all_inputs.append(scaled)

feature_acts = torch.cat(all_feature_acts, dim=0)
reconstructions = torch.cat(all_reconstructions, dim=0)
inputs = torch.cat(all_inputs, dim=0)

# Explained variance
per_token_mse = (reconstructions - inputs).pow(2).sum(dim=-1)
total_variance = (inputs - inputs.mean(0)).pow(2).sum(dim=-1)
explained_variance = 1 - per_token_mse.mean() / (total_variance.mean() + 1e-8)

# L0 and feature density
active = feature_acts.bool().float()
l0 = active.sum(-1).mean()
feature_density = active.mean(0)
dead_features = (feature_density < 1e-6).sum().item()

print(f"BatchTopK SAE Evaluation Results:")
print(f"  k (target avg active): {cfg.k}")
print(f"  L0 (actual avg active): {l0.item():.1f}")
print(f"  Explained Variance: {explained_variance.item():.4f}")
print(f"  Dead features: {dead_features}/{feature_acts.shape[-1]} ({dead_features/feature_acts.shape[-1]*100:.1f}%)")
print(f"  Mean feature density: {feature_density.mean().item():.6f}")
print(f"  TopK threshold: {sae.topk_threshold.item():.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(feature_density.cpu().numpy(), bins=50, edgecolor="black")
axes[0].set_xlabel("Feature Activation Rate")
axes[0].set_ylabel("Count")
axes[0].set_title("Feature Density Distribution (BatchTopK)")
axes[0].axvline(x=feature_density.mean().item(), color="r", linestyle="--", label="Mean")
axes[0].legend()

axes[1].hist(l0.cpu().numpy(), bins=50, edgecolor="black")
axes[1].set_xlabel("L0 (Active Features per Token)")
axes[1].set_ylabel("Count")
axes[1].set_title("L0 Distribution (BatchTopK)")
axes[1].axvline(x=l0.item(), color="r", linestyle="--", label=f"Mean={l0.item():.1f}")
axes[1].axvline(x=cfg.k, color="g", linestyle="--", alpha=0.5, label=f"Target k={cfg.k}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Logit Lens: Project Features onto Vocabulary

In [ ]:
with torch.no_grad():
    embed = model.W_dec  # T5 decoder embeddings
    projection = sae.W_dec @ embed.T  # (d_sae, d_vocab)

    top_k = 5
    vals, inds = torch.topk(projection, top_k, dim=1)

    random_indices = torch.randint(0, projection.shape[0], (10,))

    print(f"Top {top_k} tokens promoted by 10 random features:")
    print("-" * 80)
    for idx in random_indices:
        feat = idx.item()
        tokens = [model.to_string(i) for i in inds[feat]]
        scores = [f"{v:.2f}" for v in vals[feat]]
        token_strs = [f"'{t}' ({s})" for t, s in zip(tokens, scores)]
        print(f"Feature {feat:5d}: {', '.join(token_strs)}")

## Save the Trained SAE

In [ ]:
import json
from pathlib import Path
from safetensors.torch import save_file

save_dir = Path("checkpoints/batchtopk_sae_t5_large_decoder")
save_dir.mkdir(parents=True, exist_ok=True)

# Save weights
save_file(
    {
        "W_enc": sae.W_enc.data,
        "W_dec": sae.W_dec.data,
        "b_enc": sae.b_enc.data,
        "b_dec": sae.b_dec.data,
        "scaling_factor": sae.scaling_factor,
        "topk_threshold": sae.topk_threshold,
    },
    str(save_dir / "sae_weights.safetensors"),
)

# Save config
with open(save_dir / "sae_config.json", "w") as f:
    json.dump({
        "d_in": cfg.d_in,
        "d_sae": cfg.d_sae,
        "k": cfg.k,
        "hook_name": cfg.hook_name,
        "aux_loss_coefficient": cfg.aux_loss_coefficient,
        "rescale_acts_by_decoder_norm": cfg.rescale_acts_by_decoder_norm,
        "model_name": "google-t5/t5-large",
        "sae_type": "batch_topk_training_sae",
    }, f, indent=2)

# Save metrics
with open(save_dir / "training_metrics.json", "w") as f:
    json.dump(metrics, f)

print(f"Saved to {save_dir}")
print(f"  - sae_weights.safetensors")
print(f"  - sae_config.json")
print(f"  - training_metrics.json")